# Tarea 2 · CIT3360 — CNN propia (entrenada desde cero) para detección en FjordFish

**Decisiones de diseño tomadas hasta ahora**

| Aspecto | Decisión |
|---|---|
| Entrada | Imágenes reescritas a 640×640 (tamaño reducido, por ahora), sin aumento de datos ni transfer learning |
| Salida | Grilla de 20×20 celdas, 10 candidatas por celda (`N_CAND`); cada candidata: 4 valores de caja crudos (cx, cy, w, h normalizados) + 29 probabilidades de clase (28 clases + fondo) con softmax |
| Asignación | Cada caja real va a la celda de su centro y toma la candidata libre de mayor IoU; si dos peces caen en la misma celda, toman candidatas distintas (en orden de procesamiento) |
| Pérdida | Error cuadrático de caja solo en candidatas asignadas + entropía cruzada de clase; el fondo pesa `W_BG`; pesos `W_BOX`, `W_CLS`; regularización L2 |
| Entrenamiento | SGD con momentum 0,9, lr 1e-3, hasta 50 épocas, early stopping con paciencia 5 sobre la pérdida de validación |
| Memoria | Micro-batch de 3 imágenes con acumulación de gradientes (`ACCUM_STEPS`) |

**Pendiente:** definición del mAP (a acordar con el equipo), conjunto de test (el dataset solo trae train/val) e instalación del driver NVIDIA.

Recordatorio: declarar el uso de herramientas generativas según indica el enunciado.

In [1]:
import os, glob, time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Configuración de GPU (debe ejecutarse antes de usar TensorFlow por primera vez)
print("TensorFlow:", tf.__version__)
print("Build con CUDA:", tf.sysconfig.get_build_info().get("is_cuda_build"))
gpus = tf.config.list_physical_devices("GPU")
print("GPUs visibles:", gpus)
try:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
except RuntimeError as e:
    print("No se pudo activar memory_growth (reinicie el kernel y ejecute esta celda primero):", e)
if not gpus:
    print("ATENCIÓN: no se detectó GPU; el entrenamiento correrá en CPU y será muy lento.")

I0000 00:00:1793231258.573416  266938 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow: 2.22.0-rc0
Build con CUDA: True
GPUs visibles: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
### Configuración general
ROOT = "/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish"

IMG_SIZE = (640, 640)                # (alto, ancho); tamaño reducido por ahora
assert IMG_SIZE[0] == IMG_SIZE[1] and IMG_SIZE[0] % 32 == 0, "IMG_SIZE debe ser cuadrado y múltiplo de 32"
G = IMG_SIZE[0] // 32                # grilla de salida: 5 max-pools de stride 2 => stride total 32 (640 -> 20)

N_CAND = 10                          # candidatas por celda (se iterará sobre este valor)
NUM_DATASET_CLASSES = 28             # clases 0..27 del dataset (incluye "unknown")
NUM_CLASSES = NUM_DATASET_CLASSES + 1  # + fondo
BG = NUM_CLASSES - 1                 # índice de la clase fondo = 28
OUT_PER_CAND = 4 + NUM_CLASSES       # 4 de caja + 29 de clase = 33

MICRO_BATCH = 3                      # imágenes por paso (limitado por la memoria de la GPU)
ACCUM_STEPS = 5                      # micro-batches acumulados por actualización de pesos
EFFECTIVE_BATCH = MICRO_BATCH * ACCUM_STEPS
print("Batch efectivo:", EFFECTIVE_BATCH, "| grilla:", G, "x", G, "| candidatas por imagen:", G * G * N_CAND)

Batch efectivo: 15 | grilla: 20 x 20 | candidatas por imagen: 4000


In [3]:
### Datos: lectura de etiquetas YOLO y pipeline tf.data
def read_yolo_label(path):
    if not os.path.exists(path):
        return np.zeros((0, 5), dtype=np.float32)
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) == 5:
                rows.append([float(x) for x in parts])
    return np.array(rows, dtype=np.float32).reshape(-1, 5)

def list_split(split):
    img_paths = sorted(glob.glob(os.path.join(ROOT, "images", split, "*.jpg")))
    boxes = []
    for p in img_paths:
        stem = os.path.splitext(os.path.basename(p))[0]
        boxes.append(read_yolo_label(os.path.join(ROOT, "labels", split, stem + ".txt")))
    return img_paths, boxes

def make_dataset(img_paths, boxes, max_boxes, batch_size, shuffle=False):
    padded = np.full((len(boxes), max_boxes, 5), -1.0, dtype=np.float32)
    for i, b in enumerate(boxes):
        n = min(len(b), max_boxes)
        padded[i, :n] = b[:n]

    def _load(path, target):
        img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        return img, target

    ds = tf.data.Dataset.from_tensor_slices((img_paths, padded))
    if shuffle:
        ds = ds.shuffle(len(img_paths), reshuffle_each_iteration=True)
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_imgs, train_boxes = list_split("train")
val_imgs,   val_boxes   = list_split("val")
print("train:", len(train_imgs), "imágenes |", sum(len(b) for b in train_boxes), "objetos")
print("val:  ", len(val_imgs),   "imágenes |", sum(len(b) for b in val_boxes),   "objetos")

MAX_BOXES = max(len(b) for b in train_boxes + val_boxes)   # máximo de objetos en una imagen (relleno de etiquetas)
print("MAX_BOXES:", MAX_BOXES)

# Verificaciones
all_cls = np.concatenate([b[:, 0] for b in train_boxes + val_boxes if len(b)])
ids = sorted(int(i) for i in np.unique(all_cls))
print("ids de clase presentes:", ids)
assert min(ids) >= 0 and max(ids) < NUM_DATASET_CLASSES, "Hay ids de clase fuera de 0..27: revise NUM_DATASET_CLASSES"

max_in_cell = 0
for b in train_boxes + val_boxes:
    if len(b):
        cx = np.clip((b[:, 1] * G).astype(int), 0, G - 1)
        cy = np.clip((b[:, 2] * G).astype(int), 0, G - 1)
        _, counts = np.unique(cy * G + cx, return_counts=True)
        max_in_cell = max(max_in_cell, int(counts.max()))
print("máx. objetos con centro en una misma celda:", max_in_cell, "| N_CAND:", N_CAND)
if max_in_cell > N_CAND:
    print("ATENCIÓN: hay celdas con más objetos que candidatas; esas cajas quedarán sin asignar.")

sample = tf.io.decode_jpeg(tf.io.read_file(train_imgs[0]))
print("tamaño original de una imagen:", sample.shape)

train_ds = make_dataset(train_imgs, train_boxes, MAX_BOXES, MICRO_BATCH, shuffle=True)
val_ds   = make_dataset(val_imgs,   val_boxes,   MAX_BOXES, MICRO_BATCH)

train: 2086 imágenes | 4385 objetos
val:   916 imágenes | 1963 objetos
MAX_BOXES: 27
ids de clase presentes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26]
máx. objetos con centro en una misma celda: 3 | N_CAND: 10


I0000 00:00:1793231263.879339  266938 gpu_device.cc:2036] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1870 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1050, pci bus id: 0000:01:00.0, compute capability: 6.1


tamaño original de una imagen: (2160, 3840, 3)


In [4]:
## conversion de imágenes a un formato decodificable por TensorFlow (p. ej. TIFF -> PNG)
import cv2

FIRMAS = (b"\xff\xd8\xff", b"\x89PNG", b"GIF8", b"BM")   # JPEG, PNG, GIF, BMP
def formato_valido(h):
    return h.startswith(FIRMAS) or (h[:4] == b"RIFF" and h[8:12] == b"WEBP")

def asegurar_decodificable(path, cache_dir="imagenes_convertidas"):
    """Si el archivo no es un formato que TensorFlow lea (p. ej. un TIFF con extensión .jpg),
    devuelve la ruta de una copia en PNG (sin pérdida); si ya es válido, devuelve la misma ruta."""
    with open(path, "rb") as f:
        h = f.read(12)
    if formato_valido(h):
        return path
    os.makedirs(cache_dir, exist_ok=True)
    carpeta = os.path.basename(os.path.dirname(path))
    nombre = os.path.splitext(os.path.basename(path))[0]
    out = os.path.join(cache_dir, f"{carpeta}_{nombre}.png")
    if not os.path.exists(out):
        img = cv2.imread(path, cv2.IMREAD_COLOR)          # 8 bits, BGR
        if img is None:
            raise RuntimeError(f"No se pudo leer {path}")
        cv2.imwrite(out, img)
    return out

train_imgs = [asegurar_decodificable(p) for p in train_imgs]
val_imgs   = [asegurar_decodificable(p) for p in val_imgs]
print("convertidas:", sum("imagenes_convertidas" in p for p in train_imgs + val_imgs))

# Se reconstruyen los datasets con las rutas corregidas
train_ds = make_dataset(train_imgs, train_boxes, MAX_BOXES, MICRO_BATCH, shuffle=True)
val_ds   = make_dataset(val_imgs,   val_boxes,   MAX_BOXES, MICRO_BATCH)

convertidas: 12


In [5]:
### Hiperparámetros de entrenamiento
EPOCHS = 50
PATIENCE = 5                 # early stopping sobre la pérdida de validación
learning_rate = 1e-3
MOMENTUM = 0.9               # SGD con momentum
padding = "same"

# Pesos de la pérdida
W_BG      = 1/4000           # peso de cada candidata de fondo en el término de clase
W_BOX     = 3.0              # peso del error de caja
W_CLS     = 1.0              # peso del error de clase
LAMBDA_L2 = 1/10000          # regularización L2

print(tf.__version__)


2.22.0-rc0


In [6]:
### Arquitectura de la red neuronal convolucional (cuerpo)
architecture = [
    # Bloque 1
    {"type": "conv", "filters": 32, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "conv", "filters": 32, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "maxpool", "pool_size": 2, "strides": 2},
    # Bloque 2
    {"type": "conv", "filters": 64, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "conv", "filters": 64, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "maxpool", "pool_size": 2, "strides": 2},
    # Bloque 3
    {"type": "conv", "filters": 128, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "conv", "filters": 128, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "maxpool", "pool_size": 2, "strides": 2},
    # Bloque 4
    {"type": "conv", "filters": 256, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "conv", "filters": 256, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "maxpool", "pool_size": 2, "strides": 2},
    # Bloque 5
    {"type": "conv", "filters": 512, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "conv", "filters": 512, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "maxpool", "pool_size": 2, "strides": 2},
    # Bloque 6
    {"type": "conv", "filters": 1024, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "conv", "filters": 1024, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    {"type": "conv", "filters": 1024, "kernel_size": 3, "strides": 1, "activation": "relu", "padding": padding},
    # Capa de salida: N_CAND candidatas por celda, cada una con 4 (caja) + 29 (clases) valores
    {"type": "conv", "filters": N_CAND * OUT_PER_CAND, "kernel_size": 1, "strides": 1, "activation": None, "padding": padding},
]

def build_model(architecture, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    for spec in architecture:
        if spec["type"] == "conv":
            x = tf.keras.layers.Conv2D(
                spec["filters"], spec["kernel_size"],
                strides=spec["strides"], padding=spec["padding"],
                activation=spec["activation"])(x)
        elif spec["type"] == "maxpool":
            x = tf.keras.layers.MaxPooling2D(spec["pool_size"], strides=spec["strides"])(x)
    return tf.keras.Model(inputs, x)

def build_detector(architecture):
    """Cuerpo convolucional + cabeza: 4 valores de caja crudos y softmax sobre las 29 clases de cada candidata."""
    base = build_model(architecture)
    x = tf.keras.layers.Reshape((G, G, N_CAND, OUT_PER_CAND))(base.output)
    boxes = x[..., :4]                                        # (cx, cy, w, h), sin activación
    cls = tf.keras.layers.Softmax(axis=-1)(x[..., 4:])        # 28 clases + fondo
    out = tf.keras.layers.Concatenate(axis=-1)([boxes, cls])
    return tf.keras.Model(base.input, out)

### Regularización L2
def l2_regularization(model, lambda_l2):
    l2_loss = 0.0
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Conv2D):
            l2_loss += tf.reduce_sum(tf.square(layer.kernel))
    return lambda_l2 * l2_loss

In [7]:
### Asignación de cajas reales a candidatas y función de pérdida
def iou_cxcywh(a, b):
    # a, b: [..., 4] en (cx, cy, w, h). w y h negativos se truncan a 0 solo para este cálculo.
    aw, ah = tf.maximum(a[..., 2], 0.0), tf.maximum(a[..., 3], 0.0)
    bw, bh = tf.maximum(b[..., 2], 0.0), tf.maximum(b[..., 3], 0.0)
    iw = tf.maximum(tf.minimum(a[..., 0] + aw / 2, b[..., 0] + bw / 2)
                    - tf.maximum(a[..., 0] - aw / 2, b[..., 0] - bw / 2), 0.0)
    ih = tf.maximum(tf.minimum(a[..., 1] + ah / 2, b[..., 1] + bh / 2)
                    - tf.maximum(a[..., 1] - ah / 2, b[..., 1] - bh / 2), 0.0)
    inter = iw * ih
    return inter / (aw * ah + bw * bh - inter + 1e-9)

def assign_targets(pred_boxes, y_true):
    """
    pred_boxes: [B, G, G, N_CAND, 4]  cajas predichas (crudas)
    y_true:     [B, MAX_BOXES, 5]     (clase, cx, cy, w, h), relleno con -1
    Devuelve tgt_box [B, M, 4], tgt_cls [B, M], assigned [B, M], con M = G*G*N_CAND.
    Cada caja real (en orden) toma, en la celda de su centro, la candidata libre de mayor IoU.
    """
    B = tf.shape(y_true)[0]
    M = G * G * N_CAND
    pb = tf.stop_gradient(tf.reshape(pred_boxes, [B, G * G, N_CAND, 4]))

    taken   = tf.zeros([B, M], tf.float32)
    tgt_box = tf.zeros([B, M, 4], tf.float32)
    tgt_cls = tf.fill([B, M], BG)

    for k in range(MAX_BOXES):
        cls_k = y_true[:, k, 0]
        box_k = y_true[:, k, 1:5]
        valid = cls_k >= 0

        gx = tf.clip_by_value(tf.cast(box_k[:, 0] * G, tf.int32), 0, G - 1)
        gy = tf.clip_by_value(tf.cast(box_k[:, 1] * G, tf.int32), 0, G - 1)
        cell = gy * G + gx                                              # [B]

        cand_boxes = tf.gather(pb, cell, batch_dims=1)                  # [B, N, 4]
        taken_cell = tf.gather(tf.reshape(taken, [B, G * G, N_CAND]),
                               cell, batch_dims=1)                      # [B, N]

        iou = iou_cxcywh(cand_boxes, box_k[:, None, :])                 # [B, N]
        iou = tf.where(taken_cell > 0, -1.0, iou)
        best = tf.argmax(iou, axis=1, output_type=tf.int32)             # [B]
        ok = tf.logical_and(valid, tf.reduce_any(taken_cell < 1, axis=1))

        onehot = tf.one_hot(cell * N_CAND + best, M) * tf.cast(ok, tf.float32)[:, None]
        taken   = taken + onehot
        tgt_box = tgt_box + onehot[..., None] * box_k[:, None, :]
        tgt_cls = tf.where(onehot > 0, tf.cast(cls_k, tf.int32)[:, None], tgt_cls)

    return tgt_box, tgt_cls, taken

def detection_loss_sums(y_true, y_pred, w_bg):
    """Sumas SIN normalizar: (error de caja, error de clase ponderado, nº de candidatas asignadas)."""
    B = tf.shape(y_pred)[0]
    M = G * G * N_CAND
    pred = tf.reshape(y_pred, [B, M, OUT_PER_CAND])
    pred_box, pred_cls = pred[..., :4], pred[..., 4:]

    tgt_box, tgt_cls, assigned = assign_targets(y_pred[..., :4], y_true)

    box_sum = tf.reduce_sum(tf.reduce_sum(tf.square(pred_box - tgt_box), axis=-1) * assigned)
    ce = tf.keras.losses.sparse_categorical_crossentropy(tgt_cls, pred_cls)
    cls_sum = tf.reduce_sum(ce * (assigned + (1.0 - assigned) * w_bg))
    return box_sum, cls_sum, tf.reduce_sum(assigned)

def detection_loss(y_true, y_pred, w_bg, w_box, w_cls):
    """Pérdida normalizada por el número de candidatas asignadas (para diagnóstico)."""
    box_sum, cls_sum, n_pos = detection_loss_sums(y_true, y_pred, w_bg)
    n = tf.maximum(n_pos, 1.0)
    box_loss, cls_loss = box_sum / n, cls_sum / n
    return w_box * box_loss + w_cls * cls_loss, box_loss, cls_loss

In [8]:
### Modelo, optimizador y pasos de entrenamiento (con acumulación de gradientes)
# Re-ejecutar esta celda reinicia el modelo, el optimizador y los acumuladores.
model = build_detector(architecture)
print("salida del modelo:", model.output_shape, "| parámetros:", f"{model.count_params():,}")
assert model.output_shape == (None, G, G, N_CAND, OUT_PER_CAND), "forma de salida inesperada"

optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=MOMENTUM)
optimizer.build(model.trainable_variables)

acc_grads = [tf.Variable(tf.zeros(v.shape, dtype=v.dtype), trainable=False) for v in model.trainable_variables]
box_acc = tf.Variable(0.0, trainable=False)
cls_acc = tf.Variable(0.0, trainable=False)
n_acc   = tf.Variable(0.0, trainable=False)

def reset_accumulators():
    for a in acc_grads:
        a.assign(tf.zeros_like(a))
    box_acc.assign(0.0); cls_acc.assign(0.0); n_acc.assign(0.0)

@tf.function(reduce_retracing=True)
def accumulate_step(x, y):
    """Suma los gradientes de la pérdida SIN normalizar de un micro-batch."""
    with tf.GradientTape() as tape:
        pred = model(x, training=True)
        box_sum, cls_sum, n_pos = detection_loss_sums(y, pred, W_BG)
        weighted = W_BOX * box_sum + W_CLS * cls_sum
    grads = tape.gradient(weighted, model.trainable_variables)
    for a, g in zip(acc_grads, grads):
        if g is not None:
            a.assign_add(g)
    box_acc.assign_add(box_sum); cls_acc.assign_add(cls_sum); n_acc.assign_add(n_pos)

@tf.function
def apply_update():
    """Normaliza por el total de candidatas asignadas del batch efectivo, suma el gradiente L2 y actualiza."""
    n = tf.maximum(n_acc, 1.0)
    with tf.GradientTape() as tape:
        l2 = l2_regularization(model, LAMBDA_L2)
    l2_grads = tape.gradient(l2, model.trainable_variables)   # los sesgos no entran en L2 (gradiente None)
    final = [a / n + (g if g is not None else 0.0) for a, g in zip(acc_grads, l2_grads)]
    optimizer.apply_gradients(zip(final, model.trainable_variables))
    stats = (tf.identity(box_acc), tf.identity(cls_acc), tf.identity(n_acc), l2)
    for a in acc_grads:
        a.assign(tf.zeros_like(a))
    box_acc.assign(0.0); cls_acc.assign(0.0); n_acc.assign(0.0)
    return stats

@tf.function(reduce_retracing=True)
def eval_step(x, y):
    pred = model(x, training=False)
    return detection_loss_sums(y, pred, W_BG)

salida del modelo: (None, 20, 20, 10, 33) | parámetros: 28,646,506


In [8]:
### comprobacion de funcionamiento de las convoluciones 2d
x = tf.random.uniform([1, 64, 64, 3])
y = tf.keras.layers.Conv2D(8, 3, padding="same")(x)
print(y.shape)

I0000 00:00:1793220095.688822  174993 cuda_dnn.cc:442] Loaded cuDNN version 91002


(1, 64, 64, 8)


In [9]:
### Comprobación con un micro-batch antes de entrenar
xb, yb = next(iter(train_ds))
yp = model(xb, training=False)

_, _, assigned = assign_targets(yp[..., :4], yb)
print("cajas reales:", int(tf.reduce_sum(tf.cast(yb[..., 0] >= 0, tf.float32))),
      "| asignadas:", int(tf.reduce_sum(assigned)))

total, box_l, cls_l = detection_loss(yb, yp, W_BG, W_BOX, W_CLS)
print("pérdida de datos:", float(total), "(caja:", float(box_l), "| clase:", float(cls_l), ")")
print("término L2 con tu lambda:", float(l2_regularization(model, LAMBDA_L2)))

E0000 00:00:1793220109.499021  174993 cuda_executor.cc:1365] [0] Failed to allocate device memory of 2.00GiB (2147483648 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:45
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:230
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:487

W0000 00:00:1793220109.499058  174993 bfc_allocator.cc:371] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.94GiB from the lower end with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
E0000 00:00:1793220109.985634  174993 cuda_executor.cc:1365] [0] Failed to allocate device memory of 2.00GiB (2147483648 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:45
exte

cajas reales: 5 | asignadas: 5
pérdida de datos: 6.272876739501953 (caja: 0.2957213222980499 | clase: 5.385712623596191 )
término L2 con tu lambda: 0.4865734279155731


In [15]:
FIRMAS = (b"\xff\xd8\xff", b"\x89PNG", b"GIF8", b"BM")   # JPEG, PNG, GIF, BMP

def formato_valido(h):
    return h.startswith(FIRMAS) or (h[:4] == b"RIFF" and h[8:12] == b"WEBP")

for nombre, paths in [("train", train_imgs), ("val", val_imgs)]:
    malas = []
    for p in paths:
        with open(p, "rb") as f:
            h = f.read(12)
        if not formato_valido(h):
            malas.append((p, os.path.getsize(p), h))
    print(f"{nombre}: {len(malas)} archivos no reconocidos de {len(paths)}")
    for p, size, h in malas[:20]:
        print("  ", p, "| bytes:", size, "| cabecera:", h)

train: 9 archivos no reconocidos de 2086
   /home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/1775.jpg | bytes: 32776986 | cabecera: b'II*\x00\xc8!\xf4\x01\x11\x00\xfe\x00'
   /home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/1815.jpg | bytes: 23541494 | cabecera: b'II*\x00\xbc5g\x01\x13\x00\xfe\x00'
   /home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/1816.jpg | bytes: 26126140 | cabecera: b'II*\x00\xf6\xa5\x8e\x01\x15\x00\xfe\x00'
   /home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/2487.jpg | bytes: 31259496 | cabecera: b'II*\x00:\xfa\xdc\x01\x13\x00\xfe\x00'
   /home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/2489.jpg | bytes: 31259496 | cabecera: b'II*\x00:\xfa\xdc\x01\x13\x00\xfe\x00'
   /home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/2490.jpg | bytes: 36534392 | cabecera: b'II*\x00Jw-\x02\x13\x00\xfe\x00'
   /home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/2751.jpg | byte

In [16]:
malas = []
for p in train_imgs + val_imgs:
    try:
        tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
    except tf.errors.InvalidArgumentError:
        malas.append(p)
print(len(malas), "archivos con problemas", malas[:10])

W0000 00:00:1793225538.190887  174993 local_rendezvous.cc:461] Local rendezvous is aborting with status: INVALID_ARGUMENT: Unknown image file format. One of JPEG, JPEG XL, PNG, GIF, BMP, WebP required.
=== Source Location Trace: ===
tensorflow/core/kernels/image/decode_image_op.cc:281
tensorflow/core/common_runtime/single_threaded_executor.cc:449
tensorflow/core/common_runtime/function.cc:1418

W0000 00:00:1793225561.425167  174993 local_rendezvous.cc:461] Local rendezvous is aborting with status: INVALID_ARGUMENT: Unknown image file format. One of JPEG, JPEG XL, PNG, GIF, BMP, WebP required.
=== Source Location Trace: ===
tensorflow/core/kernels/image/decode_image_op.cc:281
tensorflow/core/common_runtime/single_threaded_executor.cc:449
tensorflow/core/common_runtime/function.cc:1418



12 archivos con problemas ['/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/1775.jpg', '/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/1815.jpg', '/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/1816.jpg', '/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/2487.jpg', '/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/2489.jpg', '/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/2490.jpg', '/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/2751.jpg', '/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/2840.jpg', '/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/train/2846.jpg', '/home/pisto/Desktop/Deep Learning/Tarea 2/FjordFish/images/val/2752.jpg']


In [13]:
### Entrenamiento (SGD, acumulación de gradientes, early stopping)
def epoch_loss(box_tot, cls_tot, n_tot):
    n = max(n_tot, 1.0)
    return (W_BOX * box_tot + W_CLS * cls_tot) / n, box_tot / n, cls_tot / n

history = {k: [] for k in ["train_loss", "train_box", "train_cls", "val_loss", "val_box", "val_cls", "l2", "time"]}
best_val, best_weights, wait = float("inf"), None, 0
reset_accumulators()

for epoch in range(EPOCHS):
    t0 = time.time()

    # --- entrenamiento: ACCUM_STEPS micro-batches por actualización de pesos
    tb = tc = tn = 0.0
    l2_val, pending = 0.0, 0
    for x, y in train_ds:
        accumulate_step(x, y)
        pending += 1
        if pending == ACCUM_STEPS:
            b, c, n, l2_val = [float(v) for v in apply_update()]
            tb += b; tc += c; tn += n
            pending = 0
    if pending > 0:                      # micro-batches restantes al final de la época
        b, c, n, l2_val = [float(v) for v in apply_update()]
        tb += b; tc += c; tn += n

    # --- validación
    vb = vc = vn = 0.0
    for x, y in val_ds:
        b, c, n = [float(v) for v in eval_step(x, y)]
        vb += b; vc += c; vn += n

    tr_loss, tr_box, tr_cls = epoch_loss(tb, tc, tn)
    va_loss, va_box, va_cls = epoch_loss(vb, vc, vn)
    dt = time.time() - t0
    for k, v in zip(history.keys(), [tr_loss, tr_box, tr_cls, va_loss, va_box, va_cls, l2_val, dt]):
        history[k].append(v)

    print(f"época {epoch+1:02d} | train {tr_loss:.4f} (caja {tr_box:.4f}, clase {tr_cls:.4f}, L2 {l2_val:.4f}) "
          f"| val {va_loss:.4f} (caja {va_box:.4f}, clase {va_cls:.4f}) | {dt:.0f} s")

    if va_loss < best_val:
        best_val, best_weights, wait = va_loss, model.get_weights(), 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping en la época {epoch+1}")
            break

model.set_weights(best_weights)
model.save_weights("cnn_fjordfish_best.weights.h5")
print(f"Mejor pérdida de validación: {best_val:.4f} | tiempo total de entrenamiento: {sum(history['time'])/60:.1f} min")

W0000 00:00:1793225177.809716  174993 local_rendezvous.cc:461] Local rendezvous is aborting with status: INVALID_ARGUMENT: Unknown image file format. One of JPEG, JPEG XL, PNG, GIF, BMP, WebP required.
	 [[{{node DecodeJpeg}}]]
=== Source Location Trace: ===
tensorflow/core/kernels/image/decode_image_op.cc:281
tensorflow/core/common_runtime/executor.cc:1156
tensorflow/core/kernels/data/experimental/map_and_batch_dataset_op.cc:281
tensorflow/core/data/root_dataset.cc:248
tensorflow/core/kernels/data/iterator_ops.cc:950
tensorflow/core/common_runtime/single_threaded_executor.cc:449
tensorflow/core/common_runtime/function.cc:1418



InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} Unknown image file format. One of JPEG, JPEG XL, PNG, GIF, BMP, WebP required.
	 [[{{node DecodeJpeg}}]] [Op:IteratorGetNext] name: 

In [ ]:
### Curvas de entrenamiento
ep = range(1, len(history["train_loss"]) + 1)
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, (tr, va, title) in zip(axs, [("train_loss", "val_loss", "Pérdida total (datos)"),
                                      ("train_box", "val_box", "Error de caja"),
                                      ("train_cls", "val_cls", "Error de clase")]):
    ax.plot(ep, history[tr], label="train"); ax.plot(ep, history[va], label="val")
    ax.set_title(title); ax.set_xlabel("época"); ax.legend()
plt.tight_layout()
plt.savefig("curvas_cnn.png", dpi=150)
plt.show()